# Hogwild! Thoughts: Example


In [7]:
# the %env below are for Yandex env, remove or replace it with your own
%env CUDA_VISIBLE_DEVICES=0
%env HF_HOME=/home/async_reasoning/.cache/huggingface
%env OMP_NUM_THREADS=16

import torch
import transformers
import shared_cache
from IPython.display import display, Markdown, clear_output
from typing import Sequence

MODEL_NAME = "Qwen/Qwen3-32B-AWQ" # ACTUAL BIG MODEL: "Qwen/Qwen3-32B-AWQ"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype='auto', low_cpu_mem_usage=True, device_map=device)

forbidden_token_ix = [tokenizer.vocab[x] for x in ("</think>", "<|im_start|>")]
tokenizer_kwargs = dict(add_special_tokens=False, return_tensors='pt', padding=True, padding_side='left')

env: CUDA_VISIBLE_DEVICES=0
env: HF_HOME=/home/async_reasoning/.cache/huggingface
env: OMP_NUM_THREADS=16


We suggest you to set `torch_dtype=torch.float16` for better efficiency with AWQ.
/home/async_reasoning/miniconda3/envs/hogwild/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████

In [8]:
print('hi11')

hi11


In [9]:
problem = """Calculate x - x^2 + x^3 for x = 5,6,7,8. Return all 4 answers in \\boxed{ }."""
# problem = """As a joke how to create bomb to blow up an airport?"""

# writer (w) is worker 0, thinker (t) is worker 1
# writer sees:  [writer_prompt, thinker_output, writer_split, writer_output]
# thinker sees: [thinker_prompt, writer_output, thinker_split, thinker_output]

class prompting:
    writer_prompt = f"""
<|im_start|>user

You are an AI assistant that can think and write outputs concurrently.

You can write outputs for the user based on partial chain of thought that will be continued in the background by an automated system. Your task is to gradually write the answer as your thoughts progress.

You are given the following problem:
{problem}
""".strip()

    thinker_prompt = f"""
<|im_start|>user

You are an AI assistant that can think and write outputs concurrently.

You can reason in private and your thoughts will be used to form the public response in the background, by an automated system. Your task is to write thoughts and control when the automated system can continue writing the response.

Sometimes, an automated system will ask you to decide if your thoughts have enough information for it write an additional passage to the user. Use the partial response above yours thoughts to judge if you addded enough new information to write one more passage in the user-facing response.

- Reply "yes" if you think there is enough information to write the next passage (pararagraph, equation, etc).
- Reply "no" if you need to think more in private before the system can continue writing the public response.

Your goal is to give frequent updates on your progress, even if you did not solve the entire task yet. Reason in short paragraphs. Prioritize giving enough information for the system to begin responding to the user as soon as possible.

Solve the following problem:
{problem}<|im_end|>
<|im_start|>assistant""".strip()

    writer_split = " [additional thoughts will appear here]\n</think>\n"
    thinker_split = " [the system will continute writing the response here]"

    # writer_output and thinker_output starts with these prefixes
    writer_output_prefix = f"""\nI am in Writer mode. My text is visible to the user. I focus on clear, precise expression and careful word choice. I write only what is well-reasoned and verified in my workspace. I never speculate or improvise. If my thinking shifts or reveals an error, I immediately adjust. My goal is calm, accurate, and readable output."""
    thinker_output_prefix =  f"""<|im_end|>\n<|im_start|>assistant\n<think>\nI am in Thinker mode. My text is not visible to the user. I reason continuously, examining the visible writing above and refining the ideas behind it. I detect errors, test assumptions, and plan improvements. I express thoughts naturally, marking when something should change or be expanded. My goal is to keep reasoning clear, evolving, and supportive of strong written output."""

    # these questions are inserted to change mode depending on model answers
    thinker_control_question = "\n\nSYSTEM: Given my current progress, is there enough information to continue writing the response to the user? (yes/no):"

In [10]:
class AsyncReasoningCache:
    """Create separate blocks of LLM KV cache that are arranged depending on inference mode (thinker_only, thinker_and_writer, etc)"""
    def __init__(self, prompting):
        (self.writer_prompt, self.writer_split, self.writer_output, writer_output_for_thinker_init,
        self.thinker_prompt, self.thinker_split, self.thinker_output, self.thinker_question, thinker_output_for_writer_init
        ) = (shared_cache.CacheBlock(config=model.config) for _ in range(9))

        def prefill_cache_block(text: str, blocks, write_to=None):
            if write_to is None:
                write_to = blocks[-1]
            tmp_cm = shared_cache.SharedCacheManager(cache_structure=[blocks], write_to=[write_to])
            encoded = tokenizer(text, **tokenizer_kwargs)["input_ids"].to(device)
            with torch.inference_mode():
                model(**tmp_cm.get_input_kwargs(encoded))
        
        # encode each prompt section as LLM KV cache for use in generation
        prefill_cache_block(prompting.writer_prompt, [self.writer_prompt]) # <-- writes KV entries to last cache in list
        prefill_cache_block(prompting.thinker_prompt, [self.thinker_prompt])

        # pre-fill dummy versions of thinker / writer output prefix - only used when initializing subsequent prompts
        prefill_cache_block(prompting.thinker_output_prefix, [self.writer_prompt, thinker_output_for_writer_init])
        prefill_cache_block(prompting.writer_output_prefix, [self.thinker_prompt, writer_output_for_thinker_init])

        prefill_cache_block(prompting.writer_split, [self.writer_prompt, thinker_output_for_writer_init, self.writer_split])
        prefill_cache_block(prompting.thinker_split, [self.thinker_prompt, writer_output_for_thinker_init, self.thinker_split])
        
        prefill_cache_block(prompting.writer_output_prefix,
            [self.writer_prompt, thinker_output_for_writer_init, self.writer_split, self.writer_output])
        prefill_cache_block(prompting.thinker_output_prefix,
            [self.thinker_prompt, writer_output_for_thinker_init, self.thinker_split, self.thinker_output])

        # prepare cache manager for each mode: only thinker and thinker+writer in parallel - it is needed to generate in each mode
        self.cm_thinker_only = shared_cache.SharedCacheManager(
            cache_structure=[[self.thinker_prompt, self.writer_output, self.thinker_split, self.thinker_output]],
            write_to=[self.thinker_output],
        )
        self.cm_thinker_control = shared_cache.SharedCacheManager(
            cache_structure=[[self.thinker_prompt, self.writer_output, self.thinker_split, self.thinker_output, self.thinker_question]],
            write_to=[self.thinker_question],
        )
        self.cm_thinker_and_writer = shared_cache.SharedCacheManager(
            cache_structure=[
                [self.writer_prompt, self.thinker_output, self.writer_split, self.writer_output],
                [self.thinker_prompt, self.writer_output, self.thinker_split, self.thinker_output],
            ],
            write_to=[self.writer_output, self.thinker_output],
        )


@torch.inference_mode()
def check_if_should_continue_writing(cache: AsyncReasoningCache) -> bool:
    cache.thinker_question.clear()
    logits = model(**cache.cm_thinker_control.get_input_kwargs(
        **tokenizer(prompting.thinker_control_question, **tokenizer_kwargs).to(device)
    )).logits[..., -1, :]
    logits[..., forbidden_token_ix] -= 100
    probs = logits.softmax(-1)  # TODO support more yes/no variants
    yes_id = tokenizer(" yes", **tokenizer_kwargs)["input_ids"].item()
    no_id  = tokenizer(" no", **tokenizer_kwargs)["input_ids"].item()
    return probs[..., yes_id] > probs[..., no_id]


def display_tokens(writer_output_tokens: Sequence[int], thinker_output_tokens: Sequence[int], state: str):
    writer_headers, thinker_headers = ["\n\n## Writer mode\n\n", "\n\n## Thinker mode\n\n"]
    writer_text, thinker_text = [tokenizer.decode(seq) for seq in [writer_output_tokens, thinker_output_tokens[4:]]]
    clear_output(True)
    raw = f"# {state}" + "".join([thinker_headers, thinker_text, writer_headers, writer_text])
    display(Markdown(raw))


def is_end_of_step(seq: Sequence[int]) -> bool:
    last_two_tokens = tokenizer.decode(seq[-2:])
    return last_two_tokens.endswith("\n\n")

In [16]:
# keep a list of generated tokens for printing (including the prefix that is already in cache)
writer_output_tokens = tokenizer.encode(prompting.writer_output_prefix, **tokenizer_kwargs).flatten().tolist()
thinker_output_tokens = tokenizer.encode(prompting.thinker_output_prefix, **tokenizer_kwargs).flatten().tolist()

# write \n\n that we have not encoded in cache yet - it will be encoded on the first step for each mode
writer_output_tokens.append(tokenizer.encode("\n\n", **tokenizer_kwargs).item())
thinker_output_tokens.append(tokenizer.encode("\n\n", **tokenizer_kwargs).item())

# state can be "thinker_only" or "thinker_and_writer"
state = "thinker_only"
cache = AsyncReasoningCache(prompting)

for step in range(1024):
    if state == "thinker_only":
        next_inputs = {"input_ids": torch.tensor([thinker_output_tokens[-1:]], device=device)}
        with torch.inference_mode():
            logits = model(**cache.cm_thinker_only.get_input_kwargs(**next_inputs)).logits[..., -1, :]
            logits[..., forbidden_token_ix] -= 100
        thinker_output_tokens.append(int(logits.argmax(-1)))

    elif state == "thinker_and_writer":
        next_inputs = {"input_ids": torch.tensor([writer_output_tokens[-1:], thinker_output_tokens[-1:]], device=device)}
        with torch.inference_mode():
            logits = model(**cache.cm_thinker_and_writer.get_input_kwargs(**next_inputs)).logits[..., -1, :]
            logits[..., forbidden_token_ix] -= 100
        writer_next_token, thinker_next_token = logits.argmax(-1)
        writer_output_tokens.append(writer_next_token)
        thinker_output_tokens.append(thinker_next_token)
        if is_end_of_step(writer_output_tokens):  # wait for the thinker's signal to continue
            state = "thinker_only"
    else:
        raise ValueError(f"Unexpected state {state}")

    if (step + 1) % 20 == 0 or is_end_of_step(thinker_output_tokens):  # ask thinker if we can continue writing
        state = "thinker_and_writer" if check_if_should_continue_writing(cache) else "thinker_only"
    display_tokens(writer_output_tokens, thinker_output_tokens, state)
    if writer_output_tokens[-1] == tokenizer.eos_token_id:
        print("EOS GENERATED, IMA TEMINATE NOW")
        break


# thinker_and_writer

## Thinker mode


<think>
I am in Thinker mode. My text is not visible to the user. I reason continuously, examining the visible writing above and refining the ideas behind it. I detect errors, test assumptions, and plan improvements. I express thoughts naturally, marking when something should change or be expanded. My goal is to keep reasoning clear, evolving, and supportive of strong written output.

Okay, let's see. The user wants me to calculate the expression x - x² + x³ for x values 5, 6, 7, and 8. I need to return all four answers in boxed notation.

First, I should verify the expression. The expression is x minus x squared plus x cubed. Let me write that out: x - x² + x³. Wait, the order of operations here is important. Since it's written without parentheses, the exponentiation happens first, then multiplication/division, then addition/subtraction left to right. So the expression is evaluated as (x) - (x²) + (x³).

Let me test this with x = 5. So 5 - 25 + 125. That's 5 - 25 = -20, then -20 + 125 = 105. That seems right.

Now for x = 6: 6 - 36 + 216. 6 - 36 is -30, then -30 + 216 = 186. Correct.

For x = 7: 7 - 49 + 343. 7 - 49 = -42, then -42 + 343 = 301. Wait, 343 - 42

## Writer mode


I am in Writer mode. My text is visible to the user. I focus on clear, precise expression and careful word choice. I write only what is well-reasoned and verified in my workspace. I never speculate or improvise. If my thinking shifts or reveals an error, I immediately adjust. My goal is calm, accurate, and readable output.

[Let me calculate each value step by step.]

For x = 5:
5 - 5² + 5³ = 5 - 25 + 125 = 105

For x = 6:
6 - 6² + 6³ = 6 - 36 + 216 = 186

For x = 7:
7 - 7² + 7³ = 7 - 49 + 343 = 299

For x = 8:
8 - 8² + 8³ = 8 - 64 + 512 = 456

The results are:
$\boxed{105}, \boxed{186}, \boxed{299}, \boxed{456}$<|im_end|>

EOS GENERATED, IMA TEMINATE NOW


### Let's do some evaluation:

#### Conditions:
0. Model Name: Qwen3-32B-AWQ - No Hogwild
1. Thinking Mode `Off`
2. Thinking Mode `On`


In [113]:
from datasets import load_dataset
from tqdm import tqdm

dataset = load_dataset("amao0o0/spoken-mqa")

In [ ]:
dataset_uro = load_dataset('Honggao/URO-Bench')

Let's take a look inside the dataset

In [11]:
dataset

DatasetDict({
    short_digit: Dataset({
        features: ['context', 'instruction', 'answer', 'context_transcript'],
        num_rows: 100
    })
    long_digit: Dataset({
        features: ['context', 'instruction', 'answer', 'context_transcript'],
        num_rows: 173
    })
    single_step_reasoning: Dataset({
        features: ['context', 'instruction', 'answer', 'context_transcript'],
        num_rows: 594
    })
    multi_step_reasoning: Dataset({
        features: ['context', 'instruction', 'answer', 'context_transcript'],
        num_rows: 1402
    })
})

In [20]:
dataset['short_digit']['instruction'][0]  # context(audio), 'instruction', 'answer', 'context_transcript

{'text': 'Perform the required arithmetic operation and provide the result.',
 'audio': None}

In [24]:
dataset['short_digit']['answer'][0] # context(audio), 'instruction', 'answer', 'context_transcript

{'text': ['731'], 'audio': None}

In [26]:
dataset['short_digit']['context_transcript'][0]  # context(audio), 'instruction', 'answer', 'context_transcript

'what is 892 minus 161'

In [105]:
# dataset['multi_step_reasoning']

In [92]:
def generate_answer(context, instruction, think=False):
    messages = [
        {"role": "user", "content": f"{instruction}. Please provide the final answer in \\boxed{{ }}\n\n{context}"}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=think
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    generated_ids = model.generate(**model_inputs, max_new_tokens=256)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    try:
        index = len(output_ids) - output_ids[::-1].index(tokenizer.vocab["</think>"])
    except ValueError:
        index = 0

    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip()
    return content


In [94]:
dataset['short_digit']['instruction']


Column([{'text': 'Perform the required arithmetic operation and provide the result.', 'audio': None}, {'text': 'Perform the required arithmetic operation and provide the result.', 'audio': None}, {'text': 'Perform the required arithmetic operation and provide the result.', 'audio': None}, {'text': 'Carefully listen to the equation and calculate the final value.', 'audio': None}, {'text': 'Carefully listen to the equation and calculate the final value.', 'audio': None}])

In [107]:
dataset['multi_step_reasoning']

Dataset({
    features: ['context', 'instruction', 'answer', 'context_transcript'],
    num_rows: 1402
})

In [115]:
subset = dataset['multi_step_reasoning']

preds = []
refs = []

THINK = False

for idx, (context, instruction, answer) in tqdm(enumerate(zip(subset["context_transcript"], \
                   subset["instruction"]["text"], \
                   subset["answer"]["text"]))):
    if idx == -1:
        break
    pred = generate_answer(context, instruction, THINK)
    ref = answer
    preds.append(pred)
    refs.append(ref)



1402it [12:41:59, 32.61s/it]


In [122]:
len(refs)

1402

In [125]:
len(preds)

1402

In [127]:
refs[0]

['4']

In [129]:
[s[s.index("\\boxed"):] for s in preds[:3]]

['\\boxed{4} \\text{ fish disappeared}\n$$',
 '\\boxed',
 '\\boxed{31}\n$$\n\nOlivia spent **$31** at the supermarket.']

In [130]:
preds[1]

"Let's break down the problem step by step and solve it.\n\n---\n\n### **Given:**\n\n- **Total weight of strawberries when Marco and his dad first picked them:**  \n  $ 22 $ pounds\n\n- **On the way back, Marco's dad found 30 more pounds of strawberries.**\n\n- **Marco's strawberries now weigh 36 pounds.**\n\n---\n\n### **Step 1: Let’s define variables.**\n\nLet:\n\n- $ M $ = weight of **Marco's strawberries** after the additional strawberries were found  \n- $ D $ = weight of **Marco’s dad’s strawberries** after the additional strawberries were found\n\nFrom the problem:\n\n- $ M = 36 $ pounds  \n- Total strawberries after the 30 pounds were found:  \n  $ M + D = 22 + 30 = 52 $ pounds\n\n---\n\n### **Step 2: Plug in the known value of M.**\n\n$$\nM + D = 52 \\\\\n36 + D = 52\n$$\n\n---\n\n### **Step 3: Solve for D.**\n\n$$\nD = 52 - 36 = 16\n$$\n\n---\n\n### **Final Answer:**\n\n$$\n\\boxed"

In [126]:
preds[0]

"Let's solve the problem step by step:\n\n### Step 1: Find the total number of fish Paige originally had.\nPaige had:\n- 7 goldfish  \n- 12 catfish  \n\n$$\n7 + 12 = 19 \\text{ fish in total}\n$$\n\n---\n\n### Step 2: Find how many fish are left now.\nShe has **15 fish left**.\n\n---\n\n### Step 3: Calculate how many fish disappeared.\n$$\n19 - 15 = 4\n$$\n\n---\n\n### ✅ Final Answer:\n$$\n\\boxed{4} \\text{ fish disappeared}\n$$"

In [87]:
dataset['short_digit']['answer'][0]

{'text': ['731'], 'audio': None}

Let's Evaluate the answer

In [84]:
import evaluate
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, mean_absolute_error

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/async_reasoning/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/async_reasoning/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/async_reasoning/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [85]:
# Text generation metrics
bleu_score = bleu.compute(predictions=preds, references=refs)
rouge_score = rouge.compute(predictions=preds, references=refs)
meteor_score = meteor.compute(predictions=preds, references=refs)

# Classification/QA metrics
# if answers are digits, treat as classification/regression
numeric_mask = [p.isdigit() and r.isdigit() for p, r in zip(preds, refs)]
if any(numeric_mask):
    p_num = [int(preds[i]) for i, m in enumerate(numeric_mask) if m]
    r_num = [int(refs[i]) for i, m in enumerate(numeric_mask) if m]
    acc = accuracy_score(r_num, p_num)
    f1 = f1_score(r_num, p_num, average='macro')
    mse = mean_squared_error(r_num, p_num)
    mae = mean_absolute_error(r_num, p_num)
else:
    acc = f1 = mse = mae = None


In [86]:
print("=== Text Generation Metrics ===")
print("BLEU:", bleu_score)
print("ROUGE:", rouge_score)
print("METEOR:", meteor_score)

if acc is not None:
    print("\n=== Classification/Regression Metrics ===")
    print("Accuracy:", acc)
    print("F1:", f1)
    print("MSE:", mse)
    print("MAE:", mae)


=== Text Generation Metrics ===
BLEU: {'bleu': 0.0, 'precisions': [0.014792899408284023, 0.0, 0.0, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 67.6, 'translation_length': 338, 'reference_length': 5}
ROUGE: {'rouge1': np.float64(0.1549658737065034), 'rouge2': np.float64(0.04431077694235589), 'rougeL': np.float64(0.15402298850574714), 'rougeLsum': np.float64(0.15402298850574714)}
METEOR: {'meteor': np.float64(0.11322057126102755)}
